# Pandas — «Hazlo tú» (entrega de clase)

**Nombre y apellidos: Francisco Javier Porta Borge**  

**Fecha: 26 / 10 / 2026**  

> Entrega este cuaderno en Moodle **al final de la clase**. Entregarlo acredita **asistencia**; que esté **completo** acredita **atención y seguimiento**. La valoración es **hecho / no hecho**: no se corrige acierto.


## 0. Preparación

Ejecuta esta celda **una vez**. No la modifiques: genera los datos que usarás en todos los bloques.


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000
empleados = pd.DataFrame({
    "id": np.arange(1, n + 1),
    "edad": rng.integers(18, 65, size=n),
    "salario": rng.normal(2500, 800, size=n).round(2),
    "experiencia": np.clip(rng.integers(18, 65, size=n) - rng.integers(18, 30, size=n), 0, None),
    "horas": rng.normal(40, 5, size=n).round(1),
    "nivel": rng.integers(0, 5, size=n),
    "satisfaccion": rng.normal(7, 2, size=n).round(2),
})
empleados.loc[rng.choice(n, 10, replace=False), "salario"] *= 4
empleados.loc[rng.choice(n, 15, replace=False), "satisfaccion"] = np.nan

empleados_idx = empleados.set_index("id")
print("Filas y columnas:", empleados.shape)
print(empleados.head(3))


Filas y columnas: (1000, 7)
   id  edad  salario  experiencia  horas  nivel  satisfaccion
0   1    22  3646.57            6   46.6      1          2.99
1   2    54  2573.22           23   43.3      4          5.48
2   3    48  2964.62            4   40.1      2          8.97


## Bloque 1 — Series y DataFrame

1. Selecciona las columnas `edad` y `salario` y muestra las 3 primeras filas con `head(3)`.
2. Imprime la forma (`shape`) y los tipos (`dtypes`) de esa selección.

**Resultado esperado:** 3 filas × 2 columnas; `edad` entera y `salario` decimal.


In [2]:
# TODO 1: selecciona edad y salario, muestra head(3) y luego shape/dtypes
sel = empleados[["edad", "salario"]]
print(sel.head(3))
print("Forma:", sel.shape)
print(sel.dtypes)

   edad  salario
0    22  3646.57
1    54  2573.22
2    48  2964.62
Forma: (1000, 2)
edad         int64
salario    float64
dtype: object


## Bloque 2 — Selección y filtros

Con `empleados_idx` (índice = `id`):

1. Obtén `edad` y `salario` de las etiquetas 2 a 5 (usa `.loc`).
2. Filtra los empleados con `nivel == 4` y `salario > 3000`; muestra `edad`, `salario` y `nivel`.

**Resultado esperado:** 4 filas en el apartado 1.


In [3]:
# TODO 2: usa empleados_idx.loc[2:5, [...]] y un filtro compuesto con paréntesis
# 1) .loc con etiquetas: el extremo final (5) SÍ se incluye -> 4 filas (2, 3, 4, 5)
print(empleados_idx.loc[2:5, ["edad", "salario"]])

# 2) Filtro compuesto: cada condición entre paréntesis y combinadas con &
filtro = empleados_idx[(empleados_idx["nivel"] == 4) & (empleados_idx["salario"] > 3000)]
print(filtro[["edad", "salario", "nivel"]])


    edad  salario
id               
2     54  2573.22
3     48  2964.62
4     38  2454.57
5     38  2363.67
     edad  salario  nivel
id                       
15     51  3248.44      4
34     44  3132.64      4
36     20  3034.66      4
49     49  3751.77      4
73     39  4128.06      4
84     57  3220.15      4
96     31  3568.95      4
109    40  3788.89      4
112    54  3957.04      4
139    34  3660.94      4
150    20  3788.75      4
202    60  3501.38      4
207    53  3698.70      4
220    60  3390.37      4
245    47  3668.94      4
257    49  3145.61      4
261    55  3450.18      4
278    54  3011.70      4
293    62  3580.50      4
324    61  3529.86      4
348    61  3471.35      4
366    42  5043.08      4
379    35  3033.26      4
435    38  3426.01      4
459    57  3192.77      4
523    27  3007.13      4
559    55  3041.92      4
616    48  3763.35      4
625    46  3060.58      4
669    61  3324.16      4
716    55  6070.40      4
718    41  3687.94      4
728    2

## Bloque 3 — Transformaciones

1. Crea `salario_miles = salario / 1000`.
2. Crea `tramo_horas` con `pd.cut` sobre `horas`: bins `[0, 35, 42, 60]` y etiquetas `["parcial", "completa", "extra"]`.
3. Cuenta cuántos empleados hay en cada tramo con `value_counts()`.

**Resultado esperado:** tres tramos; la suma de los recuentos es 1000.


In [4]:
# TODO 3: crea salario_miles, tramo_horas con pd.cut y usa value_counts()
empleados["salario_miles"] = empleados["salario"] / 1000
empleados["tramo_horas"] = pd.cut(
    empleados["horas"],
    bins=[0, 35, 42, 60],
    labels=["parcial", "completa", "extra"],
)
conteo = empleados["tramo_horas"].value_counts()
print(conteo)
print("Suma de recuentos:", conteo.sum())


tramo_horas
completa    501
extra       355
parcial     144
Name: count, dtype: int64
Suma de recuentos: 1000


## Bloque 4 — `groupby`

1. Si no existe, crea `senior = np.where(experiencia >= 10, "sí", "no")`.
2. Calcula la media de `salario` por `senior`.
3. Para `satisfaccion`, muestra `size()` y `count()` por `nivel` y explica la diferencia en un comentario.

**Resultado esperado:** `size()` suma 1000 y `count()` suma 985.


In [5]:
# TODO 4: crea senior si falta, media de salario por senior y size/count de satisfaccion
if "senior" not in empleados.columns:
    empleados["senior"] = np.where(empleados["experiencia"] >= 10, "sí", "no")

print(empleados.groupby("senior")["salario"].mean())

tamano = empleados.groupby("nivel")["satisfaccion"].size()
cuenta = empleados.groupby("nivel")["satisfaccion"].count()
print(pd.DataFrame({"size": tamano, "count": cuenta}))
print("Suma size:", tamano.sum(), "| Suma count:", cuenta.sum())

# size() cuenta TODAS las filas del grupo (incluidas las que tienen NaN);
# count() cuenta solo los valores NO nulos. Por eso size suma 1000 y count 985.

senior
no    2564.241929
sí    2513.362956
Name: salario, dtype: float64
       size  count
nivel             
0       187    185
1       224    222
2       208    202
3       195    192
4       186    184
Suma size: 1000 | Suma count: 985


## Bloque 5 — Ausentes

1. Cuenta los ausentes por columna con `isna().sum()`.
2. Crea `satisfaccion_imp` imputando con la mediana y comprueba que ya no quedan nulos.

**Resultado esperado:** solo `satisfaccion` tiene 15; tras imputar, 0.


In [6]:
# TODO 5: isna().sum() y satisfaccion_imp con fillna(median)
print(empleados.isna().sum())

mediana = empleados["satisfaccion"].median()
empleados["satisfaccion_imp"] = empleados["satisfaccion"].fillna(mediana)
print("Nulos tras imputar:", empleados["satisfaccion_imp"].isna().sum())

id                0
edad              0
salario           0
experiencia       0
horas             0
nivel             0
satisfaccion     15
salario_miles     0
tramo_horas       0
senior            0
dtype: int64
Nulos tras imputar: 0


## Ticket de salida

Responde en la celda siguiente, en una frase cada una:

1. Diferencia entre `.loc` y `.iloc`.
2. Qué hace `groupby`.
3. Una comparación de una línea entre R y pandas para un filtro.


In [ ]:
# 1) .loc selecciona por etiquetas del índice (y el extremo final del rango se incluye); .iloc selecciona por posición entera (y el extremo final se excluye).
# 2) groupby divide el DataFrame en grupos según los valores de una o más columnas, aplica una función (media, suma, size...) a cada grupo y combina los resultados.
# 3) R: df %>% filter(nivel == 4 & salario > 3000)  <->  pandas: df[(df["nivel"] == 4) & (df["salario"] > 3000)]
